# nb_get_watermark

Called by ADF `pl_bronze_api_payments_v3` via `act_get_watermark` (DatabricksNotebook activity).

Returns the watermark string that the pipeline passes to `updated_after` in every API call.

**Upload this notebook to:** `/Shared/ev_intelligence/bronze/nb_get_watermark`

| `load_type` | Returns |
|---|---|
| `full` | `1900-01-01T00:00:00Z` — fetches all records |
| `incremental` (audit exists) | `MAX(watermark_value)` from last succeeded run in `pipeline_audit` |
| `incremental` (no audit yet) | `1900-01-01T00:00:00Z` — safe fallback, behaves like full load |

ADF reads the return value via:
```
activity('act_get_watermark').output.runOutput
```

## Cell 1 — Receive parameters from ADF

ADF passes `pipeline_name` and `load_type` via `baseParameters`. Widgets receive them.

- `pipeline_name` scopes the audit query to only this pipeline's rows
- `load_type` decides whether to query the audit table at all

In [ ]:
dbutils.widgets.text("pipeline_name", "pl_bronze_api_payments_v3")
dbutils.widgets.text("load_type",     "incremental")

PIPELINE_NAME = dbutils.widgets.get("pipeline_name")
LOAD_TYPE     = dbutils.widgets.get("load_type")
AUDIT_TABLE   = "dbw_ev_intelligence_dev.default.pipeline_audit"
FULL_LOAD_WM  = "1900-01-01T00:00:00Z"

print(f"Pipeline  : {PIPELINE_NAME}")
print(f"Load type : {LOAD_TYPE}")
print(f"Audit table: {AUDIT_TABLE}")

## Cell 2 — Resolve watermark

For `full` load: skip the audit table entirely, return the epoch watermark so the API returns all records.

For `incremental`:
- Check if `pipeline_audit` table exists (`spark.catalog.tableExists`) — avoids error on first ever run
- Query `MAX(watermark_value)` filtered to `pipeline_name` and `status = 'succeeded'`
- If no succeeded row found yet: fall back to `1900-01-01T00:00:00Z` (safe — fetches everything, same as full load)

The watermark stored in the audit table is the `updated_after` value used in the **previous** run, not `MAX(updated_at)` from the data. This is correct — it means the next run fetches all records updated since the last run's cutoff.

In [ ]:
if LOAD_TYPE == "full":
    watermark = FULL_LOAD_WM
    print(f"Full load — using epoch watermark: {watermark}")
else:
    watermark = FULL_LOAD_WM

    if spark.catalog.tableExists(AUDIT_TABLE):
        df = spark.sql(f"""
            SELECT MAX(watermark_value) AS last_watermark
            FROM   {AUDIT_TABLE}
            WHERE  pipeline_name = '{PIPELINE_NAME}'
              AND  status        = 'succeeded'
        """)
        row = df.collect()[0]
        if row["last_watermark"] is not None:
            watermark = row["last_watermark"]
            print(f"Incremental — watermark from audit table: {watermark}")
        else:
            print(f"Incremental — no succeeded run found, using fallback: {watermark}")
    else:
        print(f"Incremental — audit table does not exist yet, using fallback: {watermark}")

print(f"Resolved watermark: {watermark}")

## Cell 3 — Return watermark to ADF

`dbutils.notebook.exit()` passes the watermark string back to ADF.

ADF reads it via `activity('act_get_watermark').output.runOutput` and stores it in `v_watermark` via `act_set_watermark`.

In [ ]:
print(f"Returning to ADF → v_watermark = {watermark}")
dbutils.notebook.exit(watermark)